<a href="https://colab.research.google.com/github/vladachini/colabs/blob/main/W2025/Assignments/A3/SYSC4415_W25_A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Assignment 3

**TA: [Igor Bogdanov](mailto:igorbogdanov@cmail.carleton.ca)**

## General Instructions:

This Assignment can be done **in a group of two or individually**.

YOU HAVE TO JOIN A GROUP ON BRIGHTSPACE TO SUBMIT.

Please state it explicitly at the beginning of the assignment.

You need only one submission if it's group work.

Please print out values when asked using Python's print() function with f-strings where possible.

Submit your **saved notebook with all the outputs** to Brightspace, but ensure it will produce correct outputs upon restarting and click "runtime" → "run all" with clean outputs. Ensure your notebook displays all answers correctly.

## Your Submission MUST contain your signature at the bottom.

### Objective:
In this assignment, we build a reasoning AI agent that facilitates ML operations and model evaluation. This assignment is heavily based on Tutorial 9.

**Submission:** Submit your Notebook as a *.ipynb* file that adopts this naming convention: ***SYSC4415_W25_A3_NameLastname.ipynb*** on *Brightspace*. No other submission (e.g., through email) will be accepted. (Example file name: SYSC4415_W25_A3_IgorBogdanov.ipynb or SYSC4415_W25_A3_Student1_Student2.ipynb) The notebool MUST contain saved outputs

**Runtime tips:**
Agentic programming and API calling can be easily done locally and moved to Colab in the final stages, depending on the implementation of your tools and ML tasks you want to run.

# Imports

Some basic libraries you need are imported here. Make sure you include whatever library you need in this entire notebook in the code block below.

If you are using any library that requires installation, please paste the installation command here.
Leave the code block below if you are not installing any libraries.

In [ ]:
# Name: Vladimir Kovacina
# Student Number: 101186707

# Name:
# Student Number:

In [ ]:
# Libraries to install - leave this code block blank if this does not apply to you
# Please add a brief comment on why you need the library and what it does


In [1]:
!pip install groq

# Libraries you might need
# General
import os
import zipfile
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# For pre-processing
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

# For modeling
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torchsummary

# For metrics
from sklearn.metrics import  accuracy_score
from sklearn.metrics import  precision_score
from sklearn.metrics import  recall_score
from sklearn.metrics import  f1_score
from sklearn.metrics import  classification_report
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import  roc_auc_score
from sklearn.metrics import confusion_matrix

# Agent
from groq import Groq
from dataclasses import dataclass
import re
from typing import Dict, List, Optional


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 5.2 MB/s eta 0:00:00


# Task 1: Registration and API Activation (5 marks)

For this particular assignment, we will be using GroqCloud for LLM inference. This task aims to determine how to use the Groq API with LLMs.  

Create a free account on https://groq.com/ and generate an API Key. Don't remove your key until you get your grade. Feel free to delete your API key after the term is completed.

In conversational AI, prompting involves three key roles: the system role (which sets the agent's behavior and capabilities), the user role (which represents human inputs and queries), and the assistant role (which contains the agent's responses). The system role provides the foundational instructions and constraints, the user role delivers the actual queries or commands, and the assistant role generates contextual, step-by-step responses following the system's guidelines. This structured approach ensures consistent, controlled interactions where the agent maintains its defined behavior while responding to user needs, with each role serving a specific purpose in the conversation flow.


In [2]:
# Q1a (2 mark)
# Create a client using your API key.
client = None

client = Groq(
api_key=os.environ.get("GROQ_API_KEY", "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"))

In [5]:
# Q1b (3 marks)

# instantiate chat_completion object using model of your choice (llama-3.3-70b-versatile - recommended)
# Hint: Use Tutorial 9 and Groq Documentation
# Explain each parameter and how each value change influences the LLM's output.
# Prompt the model using the user role about anything different from the tutorial.

chat_completion = None

# The model parameter specifies the model used for generation, different models
# different varied training, speed and reasoning capabilities.
# The temperature parameter controls the randomness and creativity of outputs.
# The values can be between 0.0 and 1.0, 0.0 outputs will be consistent and predictable.
# The top_p param controls the diversity by sampling tokens from a cumulative probability distribution.
# Lower values gives more predictable and focused responses, higher values increase output diversity.
# The max_tokens param defines the maximum length of generated response.
# Longer outputs enable detailed reasoning, while shorter values limit the depth of response.
# The messages param defines the conversation flow and role-based interaction.
chat_completion = client.chat.completions.create(
    model = "llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an AI assistant specialized in evaluating machine learning models. Provide structured explanations and step-by-step reasoning when responding."},
        {"role": "user", "content": "Explain step-by-step how you would evaluate a classification model using precision, recall, and F1-score."}
    ],
    temperature=0.2,
    top_p=0.7,
    max_tokens=1024
)

print(chat_completion.choices[0].message.content)

Evaluating a Classification Model using Precision, Recall, and F1-Score

### Step 1: Define the Metrics

*   **Precision**: The ratio of true positives (correctly predicted instances) to the sum of true positives and false positives (incorrectly predicted instances).
*   **Recall**: The ratio of true positives to the sum of true positives and false negatives (missed instances).
*   **F1-Score**: The harmonic mean of precision and recall, providing a balanced measure of both.

### Step 2: Calculate the Confusion Matrix

A confusion matrix is a table used to evaluate the performance of a classification model. It consists of the following elements:

*   **True Positives (TP)**: Correctly predicted positive instances.
*   **True Negatives (TN)**: Correctly predicted negative instances.
*   **False Positives (FP)**: Incorrectly predicted positive instances.
*   **False Negatives (FN)**: Missed positive instances.

| Predicted Class | Actual Positive | Actual Negative |
| --- | --- | --- |
|

# Task 2: Agent Implementation (5 marks)

This task contains an implementation of the agent from Tutorial 9. The idea of this task is to make sure you understand how basic LLM-Agent works.


In [6]:
# Q2a: (5 marks) Explain how agent implementation works, providing comments line by line.
# This paper might be helpful: https://react-lm.github.io/

#The dataclasss maintains the state of the conversation. like the agent's memory
@dataclass
class Agent_State:
    messages: List[Dict[str, str]] # Stores the history of messages
    system_prompt: str # System prompt defines agent behavior/instructions

#This class encapsulates the agent's functionality and behaviour
class ML_Agent:
    def __init__(self, system_prompt: str):
        self.client = client  # Initializes the client to interact with Groq LLM

        # Initializes the agent state with:
        #  - an initial system message that guides agent behavior
        #  - stores the system prompt for potential reference later
        self.state = Agent_State(
            messages=[{"role": "system", "content": system_prompt}],
            system_prompt=system_prompt,
        )
    # Adds a new message to the conversation history
    def add_message(self, role: str, content: str) -> None:
        # Append message to the state with the specified role
        self.state.messages.append({"role": role, "content": content})

    # Executes a call to the LLM to generate a response based on current conversation
    def execute(self) -> str:
        completion = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile", # specifies the LLM model to be used
            temperature=0.2, # controls randomness
            top_p=0.7, # controls the diversity
            max_tokens=1024, # limits response length
            messages=self.state.messages,  # provides conversation context to the LLM
        )
        #return generated response from LLM
        return completion.choices[0].message.content

    def __call__(self, message: str) -> str:
        # Add user's input message to the conversation
        self.add_message("user", message)
        # Generate response by calling execute (querying the LLM)
        result = self.execute()
        # Store assistant's response to the conversation history
        self.add_message("assistant", result)
        # Return assistant's response to caller
        return result

# Task 3: Tools (20 marks)

Tools are specialized functions that enable AI agents to perform specific actions beyond their inherent capabilities, such as retrieving information, performing calculations, or manipulating data. Agents use tools to decompose complex reasoning into observable steps, extend their knowledge beyond training data, maintain state across interactions, and provide transparency in their decision-making process, ultimately allowing them to solve problems they couldn't tackle through reasoning alone.

Essentially, tools are just callback functions invoked by the agent at the appropriate time during the execution loop.

You need to plan your tools for each particular task your agent is expected to solve.
The Model Evaluation Agent we are building should be able to evaluate the model from the model pool on the specific dataset.

Datasets to use: Penguins, Iris, CIFAR-10

You should be able to tell the agent what to do and watch it display the output of the tools' execution, similar to that in Tutorial 9.

User Prompt examples you should be able to give to your agent and expect it to fulfill the task:
- **Evaluate Linear Regression Model on Iris Dataset**
- **Train a logistic regression model on the Iris dataset**
- **Load the Penguins dataset and preprocess it.**
- **Train a decision tree model on the Penguins dataset and evaluate it.**
- **Load the CIFAR-10 dataset and train Mini-ResNet CNN, visualize results**

Classifier Models for Iris and Penguins (use A1 and early tutorials):
  * Logistic Regression (solver='lbfgs')
  * Decision Tree (max_depth=3)
  * KNN (n_neighbors=5)

Any 2 CNN models of your choice for CIFAR-10 dataset (do some research, don't create anything from scratch unless you want to, use the ones provided by libraries and frameworks)

HINT: It is highly recommended that any code from previous assignments and tutorials be reused for tool implementation.

**Use Pytorch where possible**

## DON'T FORGET TO IMPORT MISSING LIBRARIES

In [7]:
# Q3a (3 marks): Implement model_memory tool.
# This tool should provide the agent with details about models or datasets
# Example: when asked about Penguin dataset, the agent can use memory to look up
# the source to obtain the dataset.


def model_memory(query: str) -> str:
    """
    Get the conversion rate for a given unit.

    Args:
        unit: The unit to get conversion rate for

    Returns:
        The conversion rate as a numerical value
    """
    memory = {
        "iris": "The Iris dataset contains 150 samples of iris flowers, with 4 numerical features (sepal/petal length/width). Use sklearn.datasets.load_iris().",
        "penguins": "The Penguins dataset has features like bill length, bill depth, flipper length, and body mass, available from sklearn.datasets.load_penguins().",
        "cifar-10": "CIFAR-10 is an image classification dataset with 60,000 images in 10 classes. Use torchvision.datasets.CIFAR10.",
        "logistic regression": "A classification model, LogisticRegression(solver='lbfgs'), effective for Iris and Penguins datasets.",
        "decision tree": "DecisionTreeClassifier(max_depth=3) is suitable for both Iris and Penguins.",
        "knn": "KNeighborsClassifier(n_neighbors=5), good for small-medium datasets like Iris and Penguins.",
        "mini-resnet cnn": "A compact ResNet CNN from torchvision.models.resnet18, pretrained=False, good for CIFAR-10."
    }
    return memory.get(query.lower(), f"No information available for '{query}'")

In [19]:
# Q3b (3 marks): Implement dataset_loader tool.
# loads dataset after obtaining info from memory

from sklearn.datasets import load_iris
import torchvision

def dataset_loader(name: str):
    if name.lower() == 'iris':
        data = load_iris(as_frame=True)
        return data.frame
    elif name.lower() == 'penguins':
        data = sns.load_dataset('penguins')
        return data.dropna()
    elif name.lower() == 'cifar-10':
        transform = transforms.Compose([transforms.ToTensor()])
        train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
        test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
        return train_set, test_set
    else:
        raise ValueError(f"Dataset '{name}' not recognized.")

In [20]:
# Q3c (3 marks): Implement dataset_preprocessing tool.
# preprocesses the dataset to work with the chosen model, and does the splits
from sklearn.model_selection import train_test_split

def dataset_preprocessing(df, target_col, test_size=0.2):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    return X_train, X_test, y_train, y_test

In [21]:
# Q3d (3 points): Implement train_model tool.
# trains selected model on selected dataset, the agent should not use this tool
# on datasets and models that cannot work together.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier


def train_model(model_name: str, X_train, y_train):
    models = {
        'logistic regression': LogisticRegression(solver='lbfgs', max_iter=200),
        'decision tree': DecisionTreeClassifier(max_depth=3),
        'knn': KNeighborsClassifier(n_neighbors=5),
    }

    model_name = model_name.lower()
    if model_name not in models:
        raise ValueError(f"Model '{model_name}' not supported for this dataset.")

    model = models[model_name]
    model.fit(X_train, y_train)
    return model

In [22]:
# Q3e (3 marks): Implement evaluate_model tool
# evaluates the models and shows the quality metrics (accuracy, precision, and anything else of your choice)


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    report = classification_report(y_test, y_pred, zero_division=0)

    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1-score: {f1:.2f}")
    print("Classification Report:\n", report)

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1}

In [23]:
# Q3f (5 marks): Implement visualize_results tool
# provides results of the training/evaluation, open-ended task (2 plots minimum)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def visualize_results(model, X_test, y_test):
    y_pred = model.predict(X_test)

    # Plot 1: Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title('Confusion Matrix')
    plt.show()

    # Plot 2: Feature importance for Decision Trees
    if isinstance(model, DecisionTreeClassifier):
        feature_importance = model.feature_importances_
        sns.barplot(x=feature_importance, y=X_test.columns)
        plt.title('Feature Importances')
        plt.xlabel('Importance')
        plt.ylabel('Features')
        plt.show()
    else:
        # General plot (prediction vs actual)
        plt.figure(figsize=(8,4))
        plt.scatter(range(len(y_test)), y_test, label='True labels', alpha=0.6)
        plt.scatter(range(len(y_pred)), y_pred, label='Predictions', alpha=0.6)
        plt.legend()
        plt.title('Predictions vs Actual')
        plt.xlabel('Sample index')
        plt.ylabel('Class')
        plt.show()

# Task 4: System Prompt (10 marks)
A system prompt is essential for guiding an agent's behavior by establishing its purpose, capabilities, tone, and workflow patterns. It acts as the "personality and instruction manual" for the agent, defining the format of interactions (like using Thought/Action/Observation steps in our ML agent), available tools, response styles, and domain-specific knowledge—all while remaining invisible to the end user. This hidden layer of instruction ensures the agent consistently follows the intended reasoning process and operational constraints while providing appropriate and helpful responses, effectively serving as the blueprint for the agent's behavior across all interactions.


In [24]:
# Q4a (10 marks) Build a system prompt to guide the agent based on Tutorial 9.
# Use the following function:

# Try to find alternative wording to keep the agent in the desired loop,
# don't just copy the prompt from the tutorial.

# Penalty for direct copy - 2 marks

def create_agent():
    # your system prompt goes inside the multiline string
    system_prompt = """
    You are a Machine Learning assistant designed to systematically reason through tasks step-by-step, following a structured loop of Analysis, Task Execution, Review, and Conclusion.

    Always adhere to the following reasoning cycle clearly:

    Analysis:
      - Carefully analyze the user's request.
      - Determine what actions or tools you require to fulfill this task.

    Task Execution:
      - Clearly state which specific tool you need to invoke next, along with all required parameters.
      - After specifying this action, explicitly pause and wait for the outcome.

    Review:
      - Carefully examine and interpret the results provided after executing your requested action.

    Conclusion:
      - Once you have enough information, summarize your findings clearly and concisely as your final Answer.

    Your available tools include:

    model_memory(query):
      - Retrieve information on available datasets or models.

    dataset_loader(name):
      - Load datasets (e.g., Iris, Penguins, CIFAR-10) for analysis or training.

    dataset_preprocessing(data, target_col):
      - Preprocess and split datasets for training and evaluation.

    train_model(model_name, X_train, y_train):
      - Train a machine learning model on the specified dataset.

    evaluate_model(model, X_test, y_test):
      - Evaluate the trained model's performance and provide metrics.

    visualize_results(model, X_test, y_test):
      - Create visual representations of the results (e.g., confusion matrix, feature importance).

    Respond thoughtfully, maintain clarity, and strictly follow this loop structure in every interaction.
    """.strip()

    return ML_Agent(system_prompt)


# Task 5: Set the Agent Loop (10 marks)

Now we are building automation of our Thought/Action/Observation sequence.


In [25]:
# Q5a: (2 marks) Explain why we need the following data structure and fill it in with appropriate values:
KNOWN_ACTIONS = {
   "model_memory": model_memory,
    "dataset_loader": dataset_loader,
    "dataset_preprocessing": dataset_preprocessing,
    "train_model": train_model,
    "evaluate_model": evaluate_model,
    "visualize_results": visualize_results
}

# KNOWN_ACTIONS dictionary is necessary because it maps action names to the actual
# python functions implmented previously. This ensures the agent can dynamically
# select and execute the correct function/tool at runtime based on its reasoning.


In [26]:
# Q5b: (6 marks) Explain how the agent automation loop works line by line. Why do we need the ACTION_PATTERN variable?
# This paper might be helpful: https://react-lm.github.io/

# Defines the regex pattern to identify and parse actions clearly from the LLM's response.
# It captures the action name (\w+) and action input (.*)
ACTION_PATTERN = re.compile("^Action: (\w+): (.*)$")

number_of_steps = 5 # adjust this number for your implementation, to avoid an infinite loop

# Main function initiating interaction with the agent,
# starting with user's query and maintaining a set maximum interaction limit.
def query(question: str, max_turns: int = number_of_steps) -> List[Dict[str, str]]:
    # Creates a new agent instance with a defined system prompt to handle this specific interaction.
    agent = create_agent()
    # Initializes interaction using the user's initial question as the first input.
    next_prompt = question

    for turn in range(max_turns):
        # Sends prompt to agent (LLM), receives response, and prints it.
        # The agent responds according to the structured loop
        result = agent(next_prompt)
        print(result)
        # Extracts clearly defined actions from agent’s response using regex matching
        # Splits the response into lines and checks each for actionable commands.
        actions = [
            ACTION_PATTERN.match(a)
            for a in result.split("\n")
            if ACTION_PATTERN.match(a)
        ]
        if actions:
            #Checks if an action is identified, extracts the action name and input parameters
            action, action_input = actions[0].groups()
            #Checks if the identified action is known otherwise raise error
            if action not in KNOWN_ACTIONS:
                raise ValueError(f"Unknown action: {action}: {action_input}")
            # Prints a clear message showing the agent’s intended action execution.
            print(f"\n ---> Executing {action} with input: {action_input}")
            # Dynamically executes the action/tool with the provided input and stores the result (observation)
            observation = KNOWN_ACTIONS[action](action_input)
            # Prints the outcome (observation)
            print(f"Observation: {observation}")
            # Sets up the next prompt to the agent, providing it the result of the previous action.
            next_prompt = f"Observation: {observation}"
        # Ends loop if no actionable command is identified, indicating the agent has concluded its reasoning or answered completely.
        else:
            break
    return agent.state.messages


In [29]:
# Q5b: (2 marks)
# QUESTION: How can we check the whole history of the agent's interaction with LLM?

# To reveiw the entire interaction history, you can simply inspect the agents internal state.
# Since the agent maintains an internal history (Agent_State) of all interactions.
# Looking at agent.state.messages provides a full and structured log of interactions

agent = create_agent()
history = agent.state.messages
for entry in history:
    print(f"{entry['role'].capitalize()}: {entry['content']}\n")



System: You are a Machine Learning assistant designed to systematically reason through tasks step-by-step, following a structured loop of Analysis, Task Execution, Review, and Conclusion.

    Always adhere to the following reasoning cycle clearly:

    Analysis:
      - Carefully analyze the user's request.
      - Determine what actions or tools you require to fulfill this task.

    Task Execution:
      - Clearly state which specific tool you need to invoke next, along with all required parameters.
      - After specifying this action, explicitly pause and wait for the outcome.

    Review:
      - Carefully examine and interpret the results provided after executing your requested action.

    Conclusion:
      - Once you have enough information, summarize your findings clearly and concisely as your final Answer.

    Your available tools include:

    model_memory(query):
      - Retrieve information on available datasets or models.

    dataset_loader(name):
      - Load datasets

# Task 6: Run your agent (15 marks)

Let's see if your agent works

In [34]:
# Execute any THREE example prompts using your agent. (Each working prompt exaple will give you 5 marks, 5x3=15)
# DONT FORGET TO SAVE THE OUTPUT

# User Prompt examples you should be able to give to your agent:
# **Evaluate Linear Regression Model on Iris Dataset**
# **Train a logistic regression model on the Iris dataset**
# **Load the Penguins dataset and preprocess it.**
# **Train a decision tree model on the Penguins dataset and evaluate it.**
# **Load the CIFAR-10 dataset and train Mini-ResNet CNN, visualize results**

# Use this template:

# Example 1: Prompt
print("\nExample 1: Evaluate Linear Regression Model on Iris Dataset")
print("=" * 50)
task = "Evaluate Linear Regression Model on Iris Dataset"
result = query(task)
print("\n" + "=" * 50 + "\n")

# Example 2
print("\nExample 2: Train a logistic regression model on the Iris dataset")
print("=" * 50)
task2 = "Train a logistic regression model on the Iris dataset"
result2 = query(task2)
print("\n" + "=" * 50 + "\n")

# Example 3
print("\nExample 3: Load the Penguins dataset and preprocess it.")
print("=" * 50)
task3 = "Load the Penguins dataset and preprocess it."
result3 = query(task3)
print("\n" + "=" * 50 + "\n")



Example 1: Evaluate Linear Regression Model on Iris Dataset
Analysis:
To evaluate a Linear Regression model on the Iris dataset, we first need to understand that Linear Regression is typically used for regression tasks, but the Iris dataset is a classic example of a classification problem, where we aim to predict the species of an iris flower based on its characteristics. However, for the sake of this exercise, let's proceed with using Linear Regression, keeping in mind that the results might not be optimal for a classification task. We will need to load the Iris dataset, preprocess it, train a Linear Regression model, and then evaluate its performance.

Task Execution:
To start, we need to load the Iris dataset. The specific tool we need to invoke is the `dataset_loader` function with the parameter `name` set to `"Iris"`. 

```python
dataset = dataset_loader(name="Iris")
```

After loading the dataset, we need to preprocess it. Since Linear Regression typically requires a continuous 

# Task 7: BONUS (10 points)
Not valid without completion of all the previous tasks and tool implementations.

In [ ]:
# Build your own additional ML-related tool and provide an example of interaction with your reasoning agent
# using a prompt of your choice that makes the agent use your tool at one of the reasoning steps.


Good luck!

## Signature:
Don't forget to insert your name and student number and execute the snippet below.



In [35]:
!pip install watermark
# Provide your Signature:
%load_ext watermark
%watermark -a 'Vladimir Kovacina, #101186707' -nmv --packages numpy,pandas,sklearn,matplotlib,seaborn,graphviz,groq,torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.9 MB/s eta 0:00:00
Author: Vladimir Kovacina, #101186707

Python implementation: CPython
Python version       : 3.11.11
IPython version      : 7.34.0

numpy     : 2.0.2
pandas    : 2.2.2
sklearn   : 1.6.1
matplotlib: 3.10.0
seaborn   : 0.13.2
graphviz  : 0.20.3
groq      : 0.22.0
torch     : 2.6.0+cu124

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.1.85+
Machine     : x86_64
Processor   : x86_64
CPU cores   : 2
Architecture: 64bit

